# 고정 survey 실패1건 재개 READY — CPU
기존 source/PREPARED/plan/HEAD/pending을 고정한 재개본입니다. 원 PREPARED·quota·누적 예산을 보존하고 unknown 예약은 전액 유지합니다.
Fresh CPU runtime에서 source/input 두 작은 ZIP을 하나씩 또는 함께 업로드할 수 있습니다. Run all은 정확 preflight→명시 승계 receipt→pending1건 복구→동일 plan 재개를 수행합니다. preflight ACTION은 읽기 전용 점검만 합니다.
기존 worker exit0은 종료 증거이며 성공을 뜻하지 않습니다. 현재 lease/HEAD 충돌이나 중간 실패가 있으면 자동 반복하지 않고 회수합니다. 이 재개본은 일회성입니다.
새 pin·plan·store를 만들지 않으며 원본/캐시 업로드·GPU가 필요하지 않습니다. 시간 상한 없이 timer/status/stop,기존 자원 상한과 요청 timeout5초를 유지합니다.
원본 버전 변경·새 pending·재개 후 HEAD 진전은 이 고정 패키지로 자동 복구하지 않습니다. stop/실패/완료 뒤 마지막 회수 셀을 실행하세요.


In [ ]:
from pathlib import Path
import os,sys,time,json,uuid,hashlib,zipfile,subprocess,shutil
sys.dont_write_bytecode = True
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
ACTION = 'resume_once'  # resume_once / preflight / status / stop / recover
SOURCE_REVISION = '__ROOT_REVISION__'
SOURCE_ZIP_NAME = 'rewind_fleet_survey_resume_source_ROOT.zip'
INPUT_ZIP_NAME = 'fleet_survey_resume_inputs_ROOT.zip'
SOURCE_ZIP_SHA256 = '__ROOT_SOURCE_SHA256__'
INPUT_ZIP_SHA256 = '__ROOT_INPUT_SHA256__'
EXPECTED_RESUME_REQUEST_SHA256 = '__ROOT_RESUME_REQUEST_SHA256__'
EXPECTED_CASE_SHA256 = '1a3b1a271e6af60e5b9f43dff1561a75af4d09ec76e9147b272f259a81c56559'
LOCAL_ROOT = Path('/content/fleet_survey_resume_runtime')
SOURCE_DIR = LOCAL_ROOT / 'source'
INPUT_DIR = LOCAL_ROOT / 'inputs'
WORKSPACE = LOCAL_ROOT / 'work'
STORE = Path('/content/drive/MyDrive/REWIND-like prototype/reports/indexes/fleet_index_v1/fleet_survey_001')
PREPARED_DIR = STORE.with_name(STORE.name + '_prepared')
OWNER = 'fleet_survey_resume_001'
FAILED = None
BOOTSTRAP_OK = False
SETUP_DIAGNOSTIC = None
PREFLIGHT_FILE = None
NOTEBOOK_ERROR_CODES=frozenset(('invalid_action','root_package_pins_required','existing_bundle_sha_conflict',
'bundle_upload_cancelled','uploaded_bundle_size','uploaded_bundle_unknown_sha','uploaded_bundle_already_received',
'source_zip_sha','bundle_expansion','bundle_path','bundle_duplicate','source_reuse_exact_required','source_reuse_hash',
'indexed_gzip_1_10_3_required_restart_runtime_if_conflicting','fixed_resume_request_pin',
'fresh_runtime_required_loaded_project_module_from_other_source','need_2GiB_available_RAM_4GiB_local_disk','worker_already_running'))
def safe_notebook_failure(exc,stage):
    code=str(exc) if isinstance(exc,ValueError) and str(exc) in NOTEBOOK_ERROR_CODES else None
    internal=getattr(exc,'reason',None)
    if type(exc).__module__.startswith('rewind_') and isinstance(internal,str) and internal.isidentifier() and len(internal)<=128: code=internal
    return {'stage':stage,'type':type(exc).__name__,'reason':code}


In [ ]:
def require_source_modules(source_dir):
    source_dir=source_dir.resolve()
    for name,module in list(sys.modules.items()):
        if any(name==prefix or name.startswith(prefix+'.') for prefix in ('rewind_fleet','rewind_archive','rewind_archive_index')):
            filename=getattr(module,'__file__',None)
            if filename is None or not Path(filename).resolve().is_relative_to(source_dir):
                raise ValueError('fresh_runtime_required_loaded_project_module_from_other_source')
try:
    if ACTION not in ('resume_once','preflight','status','stop','recover'): raise ValueError('invalid_action')
    if len(SOURCE_ZIP_SHA256)!=64 or len(INPUT_ZIP_SHA256)!=64: raise ValueError('root_package_pins_required')
    LOCAL_ROOT.mkdir(parents=True,exist_ok=True)
    from google.colab import files
    def receive_verified_bundles(root,expected,upload):
        for name,digest in expected.items():
            p=root/name
            if p.exists() and (p.stat().st_size>16*1024**2 or hashlib.sha256(p.read_bytes()).hexdigest()!=digest): raise ValueError('existing_bundle_sha_conflict')
        missing={name for name in expected if not (root/name).is_file()}
        while missing:
            print('필요한 작은 ZIP (하나씩 업로드 가능):',sorted(missing))
            uploaded=upload()
            if not uploaded: raise ValueError('bundle_upload_cancelled')
            canonical_by_sha={digest:name for name,digest in expected.items()}
            for incoming_name,raw in uploaded.items():
                if len(raw)>16*1024**2: raise ValueError('uploaded_bundle_size')
                name=canonical_by_sha.get(hashlib.sha256(raw).hexdigest())
                if name is None: raise ValueError('uploaded_bundle_unknown_sha')
                if name not in missing: raise ValueError('uploaded_bundle_already_received')
                with (root/name).open('xb') as f:f.write(raw)
                missing.remove(name)
    receive_verified_bundles(LOCAL_ROOT,{SOURCE_ZIP_NAME:SOURCE_ZIP_SHA256,INPUT_ZIP_NAME:INPUT_ZIP_SHA256},files.upload)
    def bootstrap_source(archive,expected,target):
        if archive.stat().st_size>16*1024**2 or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected: raise ValueError('source_zip_sha')
        with zipfile.ZipFile(archive) as z:
            infos=z.infolist()
            if len(infos)>4096 or sum(i.file_size for i in infos)>64*1024**2: raise ValueError('bundle_expansion')
            names={}
            for i in infos:
                n=i.filename.rstrip('/')
                if not n or chr(92) in n or ':' in n or n.startswith('/') or any(x in ('','.','..') for x in n.split('/')) or (i.external_attr>>16)&0o170000==0o120000: raise ValueError('bundle_path')
                if i.is_dir(): continue
                if n.casefold() in names: raise ValueError('bundle_duplicate')
                names[n.casefold()]=(n,i)
            if target.exists():
                actual={p.relative_to(target).as_posix().casefold():p for p in target.rglob('*') if p.is_file()}
                if set(actual)!=set(names): raise ValueError('source_reuse_exact_required')
                for key,(n,i) in names.items():
                    if actual[key].is_symlink() or actual[key].read_bytes()!=z.read(i): raise ValueError('source_reuse_hash')
            else:
                target.mkdir(parents=True)
                for n,i in names.values():
                    p=target/n;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(z.read(i))
    bootstrap_source(LOCAL_ROOT/SOURCE_ZIP_NAME,SOURCE_ZIP_SHA256,SOURCE_DIR)
    if str(SOURCE_DIR) not in sys.path: sys.path.insert(0,str(SOURCE_DIR))
    require_source_modules(SOURCE_DIR)
    # package install은 source/media 비용과 분리, 요청 timeout/retry만 적용하고 전체 시간 cap은 없음.
    import importlib.util
    if importlib.util.find_spec('indexed_gzip') is None or importlib.util.find_spec('psutil') is None:
        subprocess.run([sys.executable,'-B','-m','pip','install','--no-cache-dir','--retries','0','--timeout','5','indexed_gzip==1.10.3','psutil>=5.9,<8'],check=True)
    import indexed_gzip,psutil
    if indexed_gzip.__version__!='1.10.3': raise ValueError('indexed_gzip_1_10_3_required_restart_runtime_if_conflicting')
    from rewind_fleet.survey import stage_zip,load_json
    from rewind_fleet.resume import preflight,recover_resume
    stage_zip(LOCAL_ROOT/INPUT_ZIP_NAME,INPUT_ZIP_SHA256,INPUT_DIR)
    from rewind_fleet.__main__ import main,authorized_factory
    from rewind_fleet.planning import validate
    RESUME_REQUEST=validate(load_json(INPUT_DIR/'resume_request.json'),'fleet_survey_resume_request_v1')
    if RESUME_REQUEST['content_sha256']!=EXPECTED_RESUME_REQUEST_SHA256 or RESUME_REQUEST['case']['content_sha256']!=EXPECTED_CASE_SHA256 or RESUME_REQUEST['new_source_zip_sha256']!=SOURCE_ZIP_SHA256: raise ValueError('fixed_resume_request_pin')
    PLAN_SHA256=RESUME_REQUEST['case']['plan_sha256']
    print({'old_head':RESUME_REQUEST['case']['head'],'same_plan_sha256':PLAN_SHA256,'unknown_reservation_charged_in_full':True,'time_policy':'observe_only','migration_limits':RESUME_REQUEST['limits']})
    BOOTSTRAP_OK=True
except Exception as exc:
    FAILED=safe_notebook_failure(exc,'bootstrap')
    print('준비 중단. 마지막 회수 셀을 실행하세요.',FAILED)

In [ ]:
if BOOTSTRAP_OK:
    try:
        from google.colab import auth,drive
        auth.authenticate_user();drive.mount('/content/drive')
        WORKSPACE.mkdir(parents=True,exist_ok=True)
        def survey_status():
            main(['status','--store',str(STORE)])
            if globals().get('WORKER') is not None:
                print({'worker_exit':WORKER.poll(),'wall_since_launch_seconds':time.time()-WORKER_STARTED_AT,'log':str(WORKER_LOG),'report':str(WORKER_REPORT),'error_diagnostic':str(WORKER_REPORT)+'.error.json'})
                if WORKER_REPORT.is_file():
                    r=load_json(WORKER_REPORT);print({'actual_worker_wall_seconds':r.get('seconds'),'observed_segment_seconds':r.get('observed_seconds'),'stop_reason':r.get('stop_reason')})
        def survey_stop():main(['stop','--store',str(STORE),'--plan-sha256',PLAN_SHA256])
        if ACTION=='preflight':
            PREFLIGHT_RESULT=preflight(STORE,PREPARED_DIR,RESUME_REQUEST,new_source_sha256=SOURCE_ZIP_SHA256)
            print(PREFLIGHT_RESULT)
            PREFLIGHT_FILE=WORKSPACE/('preflight-'+uuid.uuid4().hex+'.json')
            PREFLIGHT_FILE.write_text(json.dumps(PREFLIGHT_RESULT,ensure_ascii=False),encoding='utf8')
        elif ACTION=='resume_once':
            if globals().get('WORKER') is not None and WORKER.poll() is None: raise ValueError('worker_already_running')
            if psutil.virtual_memory().available<2*1024**3 or shutil.disk_usage(LOCAL_ROOT).free<4*1024**3: raise ValueError('need_2GiB_available_RAM_4GiB_local_disk')
            job=uuid.uuid4().hex;WORKER_LOG=WORKSPACE/('worker-'+job+'.log');WORKER_REPORT=WORKSPACE/('run-'+job+'.json')
            argv=[sys.executable,'-B','-u','-m','rewind_fleet.resume','--apply-once','--request',str(INPUT_DIR/'resume_request.json'),
                '--request-sha256',EXPECTED_RESUME_REQUEST_SHA256,'--new-source-sha256',SOURCE_ZIP_SHA256,'--store',str(STORE),
                '--prepared',str(PREPARED_DIR),'--workspace',str(WORKSPACE),'--owner',OWNER,'--output',str(WORKER_REPORT)]
            with WORKER_LOG.open('xb') as log:
                WORKER=subprocess.Popen(argv,cwd=SOURCE_DIR,env=dict(os.environ,PYTHONDONTWRITEBYTECODE='1'),stdout=log,stderr=subprocess.STDOUT)
            WORKER_STARTED_AT=time.time()
            print({'pid':WORKER.pid,'report':str(WORKER_REPORT),'status':'survey_status()','stop':'survey_stop()','automatic_retry':False})
            from IPython.display import display,HTML
            display(HTML('<div>Worker 시작 후 경과:0초</div><script>(function(){const start=Date.now(),node=document.currentScript.previousElementSibling;setInterval(()=>{node.textContent="Worker 시작 이후 경과:"+Math.floor((Date.now()-start)/1000)+"초 · 실행/실제 종료시간은 survey_status()";},1000);})();</script>'))
        elif ACTION=='status':survey_status()
        elif ACTION=='stop':survey_stop()
    except Exception as exc:
        FAILED=safe_notebook_failure(exc,'resume_setup')
        SETUP_DIAGNOSTIC=WORKSPACE/('resume-setup-error-'+uuid.uuid4().hex+'.json')
        WORKSPACE.mkdir(parents=True,exist_ok=True)
        SETUP_DIAGNOSTIC.write_text(json.dumps(FAILED,ensure_ascii=False),encoding='utf8')
        print('재개 준비 중단. 자동 반복하지 말고 make_recovery()로 회수하세요.',FAILED)

In [ ]:
# Worker가 실행 중이어도 이 셀의 함수는 즉시 반환합니다.
# 상태: survey_status() / 정상 중단: survey_stop()
# 이 고정 재개본은 일회성입니다. 중단/실패 후 같은 apply를 반복하지 마세요.
# 실패/중단/완료 회수: make_recovery(). Drive의 전체 pack store를 ZIP으로 복사하지 않습니다.
from pathlib import Path
import json,uuid,zipfile,hashlib
def make_recovery(download=True):
    root=globals().get('LOCAL_ROOT',Path('/content/fleet_survey_resume_runtime'));root.mkdir(parents=True,exist_ok=True)
    output=root/('fleet_survey_recovery_'+uuid.uuid4().hex+'.zip')
    if globals().get('BOOTSTRAP_OK') and callable(globals().get('recover_resume')):
        result=recover_resume(output,root=globals().get('STORE'),prepared_root=globals().get('PREPARED_DIR'),workspace=globals().get('WORKSPACE'),request_file=INPUT_DIR/'resume_request.json',diagnostic_file=globals().get('SETUP_DIAGNOSTIC') or (Path(str(WORKER_REPORT)+'.error.json') if globals().get('WORKER_REPORT') else globals().get('PREFLIGHT_FILE')))
    else:
        # bootstrap 실패에도 stdlib만으로 작은 진단을 회수합니다. 원시 예외 메시지/자격증명은 제외합니다.
        with zipfile.ZipFile(output,'x',compression=zipfile.ZIP_DEFLATED) as z:z.writestr('bootstrap_failure.json',json.dumps({'failure':globals().get('FAILED'),'source_revision':globals().get('SOURCE_REVISION'),'source_or_pack_data_included':False},ensure_ascii=False))
        result={'path':str(output),'bytes':output.stat().st_size,'sha256':hashlib.sha256(output.read_bytes()).hexdigest()}
    print(result)
    if download:
        from google.colab import files
        files.download(str(output))
    return result
if globals().get('FAILED') or globals().get('ACTION')=='recover':make_recovery()
else:print('Worker 시작 후 마지막 결과 회수는 make_recovery()를 실행하세요.')